In [ ]:
import pandas as pd
import numpy as np
import pickle
from ctf import CorrelationThresholdFilter

In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# selected_cols = [
#     'radius_mean', 
#     'texture_mean', 
#     'perimeter_mean', 
#     'area_mean', 
#     'smoothness_mean', 
#     'diagnosis'
# ]

# # Pairplot: Scatters off-diagonal, KDE distributions on diagonal
# sns.pairplot(
#     data=df[selected_cols], 
#     hue='diagnosis', 
#     palette={0: '#2ecc71', 1: '#e74c3c'},
#     diag_kind='kde',
#     corner=False  # Set to True to hide the redundant upper triangle
# )

# plt.suptitle('Pairwise Feature Relationships by Diagnosis', y=1.02)
# plt.show()

In [ ]:
df = pd.read_csv('data.csv')

In [ ]:
df.sample(15)

In [ ]:
df.info()

In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)


In [ ]:
df.describe()

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

In [ ]:
x = df.iloc[:, 1:31]
y = df.iloc[:, 0]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

pipe = Pipeline([
    ('ctf' , CorrelationThresholdFilter(0.1)),
    ('scaler', StandardScaler()), 
    ('pca' , PCA(0.95)),
    ('model' , LogisticRegression()),
])

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

pipe.fit(xtrain, ytrain)
print(f"Test Accuracy: {pipe.score(xtest, ytest) * 100:.2f}%")



In [ ]:
import joblib

joblib.dump(pipe, 'cancer_diagnosis_pipeline.joblib')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)

# 1. Generate predictions on test data
ypred = pipe.predict(xtest)

# 2. Compute raw confusion matrix
cm = confusion_matrix(ytest, ypred)
print("Confusion Matrix:\n", cm)

# 3. Print detailed classification metrics (Precision, Recall, F1-score)
print("\nClassification Report:\n")
print(
    classification_report(
        ytest, ypred, target_names=["Benign (0)", "Malignant (1)"]
    )
)

# 4. Plot visual confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Benign (0)", "Malignant (1)"]
)
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title("Confusion Matrix - Cancer Diagnosis Pipeline")
plt.grid(False)
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, x, y, cv=cv, scoring='accuracy')

print(f"Mean CV Accuracy: {scores.mean() * 100:.2f}% (+/- {scores.std() * 100:.2f}%)")